# German Day-Ahead Power Price Forecasting — Analysis Notebook

This notebook walks through the data, feature engineering, model evaluation, and
curve-translation logic interactively.

**Run the pipeline first** (`python pipeline.py`) to populate cached data in `data/raw/`.
This notebook reads from cache so you do not need live API keys to explore the results.

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.chdir('..')  # run from repo root

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load Cached Dataset

In [ ]:
cache_files = list(Path('data/raw').glob('de_all_*.parquet'))
if not cache_files:
    print('No cached data found. Run pipeline.py first.')
else:
    df_raw = pd.read_parquet(sorted(cache_files)[-1])
    print(f'Loaded {len(df_raw):,} rows, columns: {list(df_raw.columns)}')
    print(f'Date range: {df_raw.index.min()} → {df_raw.index.max()}')
    df_raw.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
local = df_raw.copy()
local.index = local.index.tz_convert('Europe/Berlin')

# Price distribution
axes[0, 0].hist(local['da_price'].dropna(), bins=80, edgecolor='none', alpha=0.75)
axes[0, 0].set_xlabel('EUR/MWh')
axes[0, 0].set_title('DA Price Distribution')

# Hourly average by hour-of-day
hourly_avg = local.groupby(local.index.hour)['da_price'].mean()
axes[0, 1].plot(hourly_avg.index, hourly_avg.values, 'o-', markersize=4)
axes[0, 1].set_xlabel('Hour of Day (CET)')
axes[0, 1].set_ylabel('Avg Price (EUR/MWh)')
axes[0, 1].set_title('Average DA Price by Hour')

# Price vs wind penetration
local['wind_total'] = local.get('wind_onshore_mw', 0).fillna(0) + local.get('wind_offshore_mw', 0).fillna(0)
local['wind_pen'] = local['wind_total'] / local['load_mw'].replace(0, np.nan)
axes[1, 0].scatter(local['wind_pen'].dropna(), local['da_price'].dropna(),
                    alpha=0.02, s=1, rasterized=True)
axes[1, 0].set_xlabel('Wind Penetration (ratio)')
axes[1, 0].set_ylabel('DA Price (EUR/MWh)')
axes[1, 0].set_title('Price vs Wind Penetration')

# Monthly average
monthly_avg = local.resample('ME')['da_price'].mean()
axes[1, 1].bar(monthly_avg.index, monthly_avg.values, width=20, alpha=0.8)
axes[1, 1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1, 1].set_ylabel('Avg Price (EUR/MWh)')
axes[1, 1].set_title('Monthly Average DA Price')
fig.autofmt_xdate()

plt.suptitle('German Day-Ahead Power Market — EDA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/05_eda_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering & Correlation

In [ ]:
from src.features.engineering import build_features, get_feature_matrix, FEATURE_COLS

df_filled = df_raw.ffill(limit=4)
df_feat = build_features(df_filled)

X, y = get_feature_matrix(df_feat)
print(f'Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features')

corr = X.corrwith(y).sort_values()
fig, ax = plt.subplots(figsize=(8, 7))
corr.plot(kind='barh', ax=ax, color=np.where(corr > 0, '#1f77b4', '#d62728'), alpha=0.8)
ax.axvline(0, color='k', linewidth=0.8)
ax.set_xlabel('Pearson r with DA Price')
ax.set_title('Feature-Target Correlations')
plt.tight_layout()
plt.savefig('outputs/figures/06_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Walk-Forward CV Results

In [ ]:
# Read pre-computed submission.csv if it exists
sub_path = Path('submission.csv')
if sub_path.exists():
    submission = pd.read_csv(sub_path, parse_dates=['id'], index_col='id')
    submission.index = submission.index.tz_localize('UTC')
    print(f'Submission: {len(submission):,} rows, {submission.index.min().date()} → {submission.index.max().date()}')
    print(submission.head())
else:
    print('submission.csv not found — run pipeline.py first')

## 5. Delivery Curve View

In [ ]:
from src.trading.curve_translation import aggregate_to_delivery, TRADING_DESK_RATIONALE

if sub_path.exists():
    monthly = aggregate_to_delivery(submission['y_pred'], period='ME')
    print('\nMonthly Delivery View:')
    print(monthly.round(2).to_string())

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(monthly.index, monthly['base_avg'], alpha=0.7, label='Base Avg')
    ax.fill_between(monthly.index, monthly['p10'], monthly['p90'], alpha=0.2, label='P10-P90')
    ax.set_ylabel('EUR/MWh')
    ax.set_title('Monthly Prompt Curve View from DA Forecast')
    ax.legend()
    plt.tight_layout()
    plt.savefig('outputs/figures/07_curve_view.png', dpi=150, bbox_inches='tight')
    plt.show()
    
print(TRADING_DESK_RATIONALE)

## 6. QA Report Review

In [ ]:
import json
qa_files = list(Path('outputs/qa_reports').glob('*.json'))
if qa_files:
    with open(sorted(qa_files)[-1]) as f:
        qa = json.load(f)
    print('=== QA Report Summary ===')
    print(f"Shape: {qa['dataset_shape']}")
    print(f"\nMissingness:")
    for col, v in qa['missingness'].items():
        print(f"  {col}: {v['n_missing']} ({v['pct_missing']}%)")
    
    if 'llm_rules' in qa:
        print(f"\nLLM-Proposed Rules:")
        for rule in qa['llm_rules'].get('llm_rules', []):
            status = '✓' if rule.get('passed') else '✗'
            print(f"  {status} [{rule['severity'].upper()}] {rule['field']}: {rule['rule']}")
else:
    print('No QA reports found — run pipeline.py first')